# Requesting 30 Instruments Historical Data Asynchronously using Data Library get_data_async and Asyncio Gather

Importing requires libraries

In [1]:
import os
import time
import asyncio
from IPython.display import Markdown, display
import lseg.data as ld
from lseg.data import session
from lseg.data._errors import LDError
from lseg.data.content import historical_pricing
from lseg.data.content.historical_pricing import Adjustments, Intervals
from dotenv import load_dotenv

Loading Platform session (Data Platform) credential from `.env` file.

In [2]:
# Load environment variables from .env file
load_dotenv(dotenv_path='.env')
# Retrieve Platform Session credentials from environment variables
app_key = os.getenv('LSEG_API_KEY')
username = os.getenv('LSEG_MACHINE_ID')
password = os.getenv('LSEG_PASSWORD')

Define required variables.

In [3]:
# -- Top 30 S&P 500 companies ----------------------------------------------------
INSTRUMENTS = {
    "NVDA.O":  "NVIDIA",
    "AAPL.O":  "Apple",
    "MSFT.O":  "Microsoft",
    "AMZN.O":  "Amazon",
    "GOOG.O":  "Alphabet",
    "AVGO.O":  "Broadcom",
    "META.O":  "Meta",
    "ORCL.N":  "Oracle",
    "IBM.N":   "IBM",
    "PLTR.O":  "Palantir",
    "NFLX.O":  "Netflix",
    "TSLA.O":  "Tesla",
    "CRM.N":   "Salesforce",
    "AMD.O":   "AMD",
    "INTC.O":  "Intel",
    "ARM.O":   "Arm Holdings",
    "TXN.O": "Texas Instruments",
    "CSCO.O":  "Cisco Systems",
    "WMT.O":   "Walmart",
    "LLY.N":   "Eli Lilly and Company",
    "JPM.N":   "JPMorgan Chase & Co.",
    "XOM.N":   "Exxon Mobil Corporation",
    "V.N":     "Visa Inc.",
    "JNJ.N":   "Johnson & Johnson",
    "MU.O":    "Micron Technology, Inc.",
    "MA.N":    "Mastercard Incorporated",
    "COST.O":  "Costco Wholesale Corporation",
    "CVX.N":   "Chevron Corporation",
    "BAC.N":   "Bank of America Corporation",
    "CAT.N":   "Caterpillar Inc.",
}

# ── Date range ─────────────────────────────────────────────────────────────────
START = "2025-11-01T00:00:00Z"
END   = "2026-02-28T23:59:59Z"

# ── Event correction filters ───────────────────────────────────────────────────
EVENT_ADJUSTMENTS = [Adjustments.EXCHANGE_CORRECTION,
                    Adjustments.MANUAL_CORRECTION,
                    Adjustments.CCH,
                    Adjustments.CRE,
                    Adjustments.RPO,
                    Adjustments.RTS]

# ── Field lists ─────────────────────────────────────────────────────────────────
INTERDAY_FIELDS = ["BID", "ASK", "OPEN_PRC", "HIGH_1", "LOW_1", "TRDPRC_1", "NUM_MOVES", "TRNOVR_UNS"]

Intitalize and open the session.

In [4]:
# Create a platform session with provided credentials for authentication
ld_session = session.platform.Definition(
         app_key=app_key,
         grant=session.platform.GrantPassword(
             username=username,
             password=password
         ),
         signon_control=True
).get_session()

# Set this session as the default for all subsequent Data Library calls
session.set_default(ld_session)

# Open the connection to the LSEG Data Platform
ld_session.open()

<OpenState.Opened: 'Opened'>

Define `display_response` that handles `asyncio.gather` with `return_exceptions=True`.

In [5]:
def display_response(data, format="df"):
    """Display the result of each async API call.

    For each response:
    - Prints the exception message if the task raised a Python error.
    - Warns if the response has no closure label (unexpected type).
    - Renders the DataFrame on a successful HTTP response.
    - Prints the HTTP status code on a failed (4xx/5xx) response.
    """
    for api_response in data:
        # Task raised a Python exception (e.g. network error, timeout)
        if isinstance(api_response, Exception):
            print(f"\nTask failed with exception: {api_response}")
            continue

        # Guard against unexpected response types
        if not hasattr(api_response, 'closure'):
            print(f"\nUnexpected response type: {type(api_response)}")
            continue

        print(f"\nResponse received for: {api_response.closure}")

        if api_response.is_success:
            if format == "df":
                display(api_response.data.df)
            elif format == "json":
                display(api_response.data.raw)
        else:
            # HTTP-level failure (4xx / 5xx from the platform)
            print(f"Request failed — HTTP status: {api_response.http_status}")

Requests 30 Instruments asynchronously with time counter. The code uses `response.data.raw` statement to display the raw output data in JSON format. If you instead use `response.data.df` statement to retrieve the data as a DataFrame, please note that additional time is required to convert the JSON data into a DataFrame.

In [6]:
rics = list(INSTRUMENTS.keys())

# Convert the company dictionary to a list once.
# Example item: ("NVDA.O", "NVIDIA")
list_of_rics_companies = list(INSTRUMENTS.items())

display(Markdown("**Start the wall-clock timer...**"))
start_time = time.perf_counter()

try:
    historical_data = await asyncio.gather(  # pylint: disable=await-outside-async
        *[
            historical_pricing.summaries.Definition(
                universe=ric,
                fields=INTERDAY_FIELDS,
                interval=Intervals.DAILY,
                start=START,
                end=END,
                adjustments=EVENT_ADJUSTMENTS,
            ).get_data_async(
                closure=f"{company} Interday Historical Summaries (Daily)"
            )
            for ric, company in list_of_rics_companies
        ],
        return_exceptions=True,
    )

    display(Markdown(f"**({len(rics)}) Companies Historical Price Interday Historical Summaries (Daily) via Iteration**"))
    display_response(historical_data, format="json")
except* LDError as errors:
    for error in errors.exceptions:
        print(error)
finally:
    elapsed = time.perf_counter() - start_time
    elapsed_string = f"**Sending Historical-Pricing Interday Historical Summaries (Daily) for ({len(rics)}) RICs in {elapsed:0.2f} seconds.**"
    display(Markdown(elapsed_string))

**Start the wall-clock timer...**

**(30) Companies Historical Price Interday Historical Summaries (Daily) via Iteration**


Response received for: NVIDIA Interday Historical Summaries (Daily)


{'universe': {'ric': 'NVDA.O'},
 'interval': 'P1D',
 'summaryTimestampLabel': 'endPeriod',
 'adjustments': ['exchangeCorrection',
  'manualCorrection',
  'CCH',
  'CRE',
  'RTS',
  'RPO'],
 'defaultPricingField': 'TRDPRC_1',
 'headers': [{'name': 'DATE', 'type': 'string'},
  {'name': 'TRDPRC_1', 'type': 'number', 'decimalChar': '.'},
  {'name': 'HIGH_1', 'type': 'number', 'decimalChar': '.'},
  {'name': 'LOW_1', 'type': 'number', 'decimalChar': '.'},
  {'name': 'OPEN_PRC', 'type': 'number', 'decimalChar': '.'},
  {'name': 'BID', 'type': 'number', 'decimalChar': '.'},
  {'name': 'ASK', 'type': 'number', 'decimalChar': '.'},
  {'name': 'TRNOVR_UNS', 'type': 'number', 'decimalChar': '.'},
  {'name': 'NUM_MOVES', 'type': 'number', 'decimalChar': '.'}],
 'data': [['2026-02-27',
   177.19,
   182.59,
   176.38,
   181.25,
   177.14,
   177.15,
   55882286062,
   3823737],
  ['2026-02-26',
   184.89,
   194.29,
   184.315,
   194.27,
   184.87,
   184.89,
   67579404866,
   4877577],
  ['2026


Response received for: Apple Interday Historical Summaries (Daily)


...


Response received for: Bank of America Corporation Interday Historical Summaries (Daily)


{'universe': {'ric': 'BAC.N'},
 'interval': 'P1D',
 'summaryTimestampLabel': 'endPeriod',
 'adjustments': ['exchangeCorrection',
  'manualCorrection',
  'CCH',
  'CRE',
  'RTS',
  'RPO'],
 'defaultPricingField': 'TRDPRC_1',
 'headers': [{'name': 'DATE', 'type': 'string'},
  {'name': 'TRDPRC_1', 'type': 'number', 'decimalChar': '.'},
  {'name': 'HIGH_1', 'type': 'number', 'decimalChar': '.'},
  {'name': 'LOW_1', 'type': 'number', 'decimalChar': '.'},
  {'name': 'OPEN_PRC', 'type': 'number', 'decimalChar': '.'},
  {'name': 'BID', 'type': 'number', 'decimalChar': '.'},
  {'name': 'ASK', 'type': 'number', 'decimalChar': '.'},
  {'name': 'TRNOVR_UNS', 'type': 'number', 'decimalChar': '.'},
  {'name': 'NUM_MOVES', 'type': 'number', 'decimalChar': '.'}],
 'data': [['2026-02-27',
   49.83,
   51.41,
   49.32,
   51.34,
   49.8,
   49.81,
   1506147994,
   61761],
  ['2026-02-26', 52.3, 52.67, 51.75, 51.88, 52.29, 52.3, 1451883232, 42276],
  ['2026-02-25', 51.69, 51.76, 50.43, 50.6, 51.7, 51.71


Response received for: Caterpillar Inc. Interday Historical Summaries (Daily)


{'universe': {'ric': 'CAT.N'},
 'interval': 'P1D',
 'summaryTimestampLabel': 'endPeriod',
 'adjustments': ['exchangeCorrection',
  'manualCorrection',
  'CCH',
  'CRE',
  'RTS',
  'RPO'],
 'defaultPricingField': 'TRDPRC_1',
 'headers': [{'name': 'DATE', 'type': 'string'},
  {'name': 'TRDPRC_1', 'type': 'number', 'decimalChar': '.'},
  {'name': 'HIGH_1', 'type': 'number', 'decimalChar': '.'},
  {'name': 'LOW_1', 'type': 'number', 'decimalChar': '.'},
  {'name': 'OPEN_PRC', 'type': 'number', 'decimalChar': '.'},
  {'name': 'BID', 'type': 'number', 'decimalChar': '.'},
  {'name': 'ASK', 'type': 'number', 'decimalChar': '.'},
  {'name': 'TRNOVR_UNS', 'type': 'number', 'decimalChar': '.'},
  {'name': 'NUM_MOVES', 'type': 'number', 'decimalChar': '.'}],
 'data': [['2026-02-27',
   742.83,
   751.49,
   731.29,
   745.22,
   742.84,
   743.14,
   786363021,
   8767],
  ['2026-02-26',
   752.93,
   767.09,
   728.47,
   763.83,
   752.73,
   752.74,
   592145453,
   13544],
  ['2026-02-25',
  

**Sending Historical-Pricing Interday Historical Summaries (Daily) for (30) RICs in 11.50 seconds.**

Closing the session.

In [7]:
# Close the session to release resources; in a long-running application, consider keeping the session open and reusing it for subsequent API calls instead.
ld_session.close()
ld.close_session()